# Phase 8 - Training (OpenML version)

## Reliable EMNIST download via scikit-learn's fetch_openml

The `emnist` pip library's download is broken (it fetches from an unreliable
university server - you saw it grab 120 kB then fail with BadZipFile). This
notebook gets the SAME data from **OpenML** using `fetch_openml` - the exact
mechanism that already downloaded MNIST successfully on your machine.

No TensorFlow, no broken zip, coexists with MediaPipe.

This notebook only does the training and saves `char_model.joblib`. Your live
app (`sentence_writer.py` from the Phase 8 notebook) is unchanged - it just
loads that file.

## 1. Download EMNIST from OpenML

We try the balanced EMNIST set. The download is cached after the first run.
This is a large dataset, so give it a few minutes.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

# Try by name first, fall back to the numeric data_id if the name resolves wrong.
print("Downloading EMNIST from OpenML (cached after first run)...")
try:
    ds = fetch_openml("EMNIST_Balanced", version=1, as_frame=False)
except Exception as e:
    print("Name lookup failed, trying data_id=41039 ...", str(e)[:80])
    ds = fetch_openml(data_id=41039, as_frame=False)

X = ds.data.astype(np.float32)
y = ds.target.astype(int)
print("Downloaded:", X.shape, "labels:", y.shape, "| classes:", len(np.unique(y)))

Downloaded: (131600, 784) labels: (131600,) | classes: 47


## 2. CONFIRM ORIENTATION before training (important!)

EMNIST images are often stored transposed. Rather than assume, we display a few
samples so YOU can see whether they look upright. Run this cell and look at the
printed characters.

If they look rotated/mirrored, we apply the fix in the next cell. This check is
here because the exact orientation can differ between dataset sources, and
training on wrongly-oriented images gives useless predictions.

In [2]:
import numpy as np

LABEL_MAP = ("0123456789"
             "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
             "abdefghnqrt")

def ascii_show(img28):
    """Print a 28x28 image as ASCII so we can eyeball orientation in a notebook."""
    chars = " .:-=+*#%@"
    out = []
    for row in img28:
        out.append("".join(chars[min(9, int(v/28))] for v in row))
    return "\n".join(out)

# show the first sample RAW (no fix) and its label
idx = 0
raw = X[idx].reshape(28, 28)
print("Label:", LABEL_MAP[y[idx]], "(class", y[idx], ")")
print("RAW orientation:")
print(ascii_show(raw))

Label: r (class 45 )
RAW orientation:
                            
                            
         ..                 
        .==:                
       :+%%%+-.             
       =#@@@%%#+            
       =#@@@@@%#:           
       .=%%@@@@%#-.         
        :*#%@@@@%+-.        
          .+%@@@@%%*=       
         :=#@@@@@@@%#.      
         =*%@@@@@@@%#.      
        :#%@@@@%@@@*-       
       .=%@@%%*+*##-.       
       :*@@@%*-.:==:        
       =#@@@*-              
       =#@@%=.              
       =#@@%:               
       =#@@#.               
       =#@@#.               
       =#@@#.               
       =#@@#.               
       =#@@#.               
       .=%%+                
         :-.                
                            
                            
                            


## 3. Apply orientation fix and re-check

Run this to see the same sample with the transpose fix applied. Compare with
the raw version above - whichever looks like a correct, upright character is
the one to train on. The code keeps the fixed version; if the RAW above already
looked correct, set `APPLY_FIX = False`.

In [3]:
APPLY_FIX = True   # set False if the RAW image above already looked upright

def fix_orientation(flat):
    img = flat.reshape(28, 28)
    return np.fliplr(np.rot90(img, 3)).reshape(-1)

if APPLY_FIX:
    fixed = fix_orientation(X[idx])
    print("Label:", LABEL_MAP[y[idx]], "- FIXED orientation:")
    print(ascii_show(fixed.reshape(28, 28)))
    print("\nIf THIS looks upright and matches the label, keep APPLY_FIX=True.")
else:
    print("APPLY_FIX is False - will train on raw images.")

Label: r - FIXED orientation:
                            
                            
                            
                            
                            
                            
                            
    :==.     .:========.    
   .+##=:   :=*########=    
  .=%@@%* :=#%@@@@@@@@@%:   
  .=%@@%#.=*%@@@@@@@@@@%-   
   :%@@@%+#%@@@@%%#####+.   
    +%@@@%@@@%%*=:.....     
    -%@@@@@@@%*-.           
    .#%@@@@@@*-             
     +#%@@@@%+.             
      :#%@@@@*:             
       -+%@@@#=             
       .-%@@@#=             
        .*%%*-:             
         =##-.              
          ..                
                            
                            
                            
                            
                            
                            

If THIS looks upright and matches the label, keep APPLY_FIX=True.


## 4. Train (only after orientation looks correct)

Applies the orientation choice to all images, normalises, trains the MLP, and
saves `char_model.joblib` in the same format the live app expects.

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
import joblib

Xp = X.copy()
if APPLY_FIX:
    Xp = np.array([fix_orientation(row) for row in Xp])
Xp = Xp / 255.0

X_tr, X_te, y_tr, y_te = train_test_split(Xp, y, test_size=0.1, random_state=42)

print("Training MLP on", X_tr.shape[0], "samples (a few minutes)...")
clf = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=40,
                    early_stopping=True, random_state=42, verbose=True)
clf.fit(X_tr, y_tr)

acc = clf.score(X_te, y_te)
print(f"\nTest accuracy: {acc:.3f}")

joblib.dump({"model": clf, "label_map": LABEL_MAP}, "char_model.joblib")
print("Saved char_model.joblib - ready for sentence_writer.py")

Training MLP on 118440 samples (a few minutes)...
Iteration 1, loss = 1.18182300
Validation score: 0.761229
Iteration 2, loss = 0.67555662
Validation score: 0.807835
Iteration 3, loss = 0.54956978
Validation score: 0.825650
Iteration 4, loss = 0.48462647
Validation score: 0.831138
Iteration 5, loss = 0.44084451
Validation score: 0.832658
Iteration 6, loss = 0.40967732
Validation score: 0.836035
Iteration 7, loss = 0.38315300
Validation score: 0.840088
Iteration 8, loss = 0.35846072
Validation score: 0.838315
Iteration 9, loss = 0.34025884
Validation score: 0.842367
Iteration 10, loss = 0.32268006
Validation score: 0.846927
Iteration 11, loss = 0.30740171
Validation score: 0.843465
Iteration 12, loss = 0.29434788
Validation score: 0.841692
Iteration 13, loss = 0.28162254
Validation score: 0.844140
Iteration 14, loss = 0.26881570
Validation score: 0.844900
Iteration 15, loss = 0.25891277
Validation score: 0.842367
Iteration 16, loss = 0.24726946
Validation score: 0.848615
Iteration 17, l

## 5. Verify predictions

Predicts on held-out samples and shows predicted vs actual characters. If these
mostly match, the model is good and the live app will work.

In [5]:
bundle = joblib.load("char_model.joblib")
clf2, lmap = bundle["model"], bundle["label_map"]
preds = clf2.predict(X_te[:12])
print("Predicted:", [lmap[p] for p in preds])
print("Actual   :", [lmap[t] for t in y_te[:12]])
matches = sum(p == t for p, t in zip(preds, y_te[:12]))
print(f"{matches}/12 correct on this sample")

Predicted: ['C', '6', 'W', 'f', 'r', '8', 'h', 'O', '2', 'F', 'U', 'Y']
Actual   : ['C', '6', 'N', 'f', 'r', 'q', 'h', 'O', '2', 'F', 'U', 'Y']
10/12 correct on this sample


## Done

`char_model.joblib` is saved. Run your `sentence_writer.py` (from the main
Phase 8 notebook) to write in the air - it loads this model unchanged.

If orientation looked wrong and flipping `APPLY_FIX` didn't fix it, tell me what
the ASCII images looked like and I'll adjust the transform.

In [6]:
script = r"""
# sentence_writer.py - build sentences in the air, one confirmed character at a time.
import time, platform
import cv2
import numpy as np
import mediapipe as mp
import joblib
import pyautogui

try:
    bundle = joblib.load("char_model.joblib")
    MODEL = bundle["model"]
    LABEL_MAP = bundle["label_map"]
except Exception as e:
    raise SystemExit("Could not load char_model.joblib - run the training "
                     "notebook first. Error: " + str(e))

WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP = 4, 8, 12
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]
import math

def fingers_up(lm):
    f = [1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0]
    for tip, pip in zip(TIPS[1:], PIPS[1:]):
        f.append(1 if lm[tip].y < lm[pip].y else 0)
    return f

def hand_size(lm):
    return math.hypot(lm[5].x - lm[WRIST].x, lm[5].y - lm[WRIST].y) + 1e-6

def pinch_ratio(lm):
    d = math.hypot(lm[INDEX_TIP].x - lm[THUMB_TIP].x,
                   lm[INDEX_TIP].y - lm[THUMB_TIP].y)
    return d / hand_size(lm)

def preprocess(canvas):
    ys, xs = np.where(canvas > 0)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    crop = canvas[y0:y1+1, x0:x1+1]
    h, w = crop.shape
    side = max(h, w)
    sq = np.zeros((side, side), np.uint8)
    sq[(side-h)//2:(side-h)//2+h, (side-w)//2:(side-w)//2+w] = crop
    small = cv2.resize(sq, (20, 20), interpolation=cv2.INTER_AREA)
    img = np.zeros((28, 28), np.uint8)
    img[4:24, 4:24] = small
    return (img.astype(np.float32) / 255.0).reshape(1, 784)

def main():
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) if platform.system()=="Windows" else cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    hands = mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=1,
                                     min_detection_confidence=0.7, min_tracking_confidence=0.5)
    draw_util = mp.solutions.drawing_utils

    canvas = None
    prev_pt = None
    sentence = ""
    cooldown = 0
    COOLDOWN_FRAMES = 18

    print("Index=draw | palm=recognise | pinch=space | two=backspace | fist=clear | q=quit")
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        if canvas is None:
            canvas = np.zeros((h, w), np.uint8)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res = hands.process(rgb)
        rgb.flags.writeable = True

        status = "show hand"
        if cooldown > 0:
            cooldown -= 1

        if res.multi_hand_landmarks:
            hand = res.multi_hand_landmarks[0]
            draw_util.draw_landmarks(frame, hand, mp.solutions.hands.HAND_CONNECTIONS)
            lm = hand.landmark
            f = fingers_up(lm)
            total = sum(f)
            ratio = pinch_ratio(lm)

            if total == 0:                       # fist -> clear current char
                canvas[:] = 0; prev_pt = None
                status = "CLEARED char"
            elif ratio < 0.18 and cooldown == 0: # pinch -> space
                sentence += " "
                pyautogui.typewrite(" ")
                cooldown = COOLDOWN_FRAMES
                status = "SPACE"
            elif total == 2 and f[1] == 1 and f[2] == 1 and cooldown == 0:  # two -> backspace
                if sentence:
                    sentence = sentence[:-1]
                    pyautogui.press("backspace")
                cooldown = COOLDOWN_FRAMES
                status = "BACKSPACE"
            elif total == 5 and cooldown == 0:   # palm -> recognise+append
                vec = preprocess(canvas)
                if vec is not None:
                    pred = int(MODEL.predict(vec)[0])
                    ch = LABEL_MAP[pred]
                    sentence += ch
                    pyautogui.typewrite(ch)
                    canvas[:] = 0; prev_pt = None
                    cooldown = COOLDOWN_FRAMES
                    status = f"ADDED: {ch}"
                else:
                    status = "nothing drawn"
            else:
                index_only = f[1] == 1 and f[2] == 0 and ratio >= 0.18
                if index_only:
                    cx, cy = int(lm[INDEX_TIP].x*w), int(lm[INDEX_TIP].y*h)
                    if prev_pt is not None:
                        cv2.line(canvas, prev_pt, (cx, cy), 255, 16)
                    prev_pt = (cx, cy)
                    status = "DRAWING"
                else:
                    prev_pt = None

        # overlay strokes
        colour = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
        colour[:, :, 0] = 0
        frame = cv2.addWeighted(frame, 1.0, colour, 0.8, 0)

        cv2.putText(frame, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        # sentence bar at the bottom
        cv2.rectangle(frame, (0, h-40), (w, h), (40,40,40), -1)
        cv2.putText(frame, "> " + sentence[-40:], (10, h-12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.imshow("Sentence Air Writer", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release(); cv2.destroyAllWindows(); hands.close()
    print("Final sentence:", sentence)

if __name__ == "__main__":
    main()
"""
with open("sentence_writer.py", "w") as fh:
    fh.write(script)
print("Wrote sentence_writer.py - run with: python sentence_writer.py")

Wrote sentence_writer.py - run with: python sentence_writer.py
